# Let's play ping pong

From one side, I will run this part including my GCN credentials as 'Producer', but streaming on the test version of GCN.

In [ ]:
from gcn_kafka import Producer
import base64
import json

In [ ]:
# Test
producer = Producer(config={'message.max.bytes': int(5 * 1024 * 1024)}, client_id='..', client_secret='..', domain='test.gcn.nasa.gov')

In [ ]:
with open('./guano.loc_map_758917906.json') as f:
    exnotice=json.load(f)

In [ ]:
exnotice

In [ ]:
with open('./771151674_0_n_PROBMAP.fits', 'rb') as f:
    skymap_bytes = f.read()

In [ ]:
enconded_map = base64.b64encode(skymap_bytes)

In [ ]:
exnotice['healpix_file'] = enconded_map.decode()
example_str = json.dumps(exnotice)

In [ ]:
len(example_str)/1024/1024  # size in MB

In [ ]:
producer.produce('gcn.notices.swift.bat.guano',example_str)
producer.flush()

On the other side, who will listen will have to set up the following. The credentials need to be created on https://test.gcn.nasa.gov/user/credentials

In [ ]:
from gcn_kafka import Consumer

consumer = Consumer(client_id='..',
                    client_secret='..',
                    domain='test.gcn.nasa.gov')

# Subscribe to topics and receive alerts
consumer.subscribe(['gcn.notices.swift.bat.guano'])
while True:
    for message in consumer.consume(timeout=1):
        if message.error():
            print(message.error())
            continue
        # Print the topic and message ID
        print(f'topic={message.topic()}, offset={message.offset()}')
        value = message.value()
        print(value)